# Inside a run: tracking modde's state

modde exposes its complete state, so you can follow what an algorithm does,
generation by generation or even step by step. This notebook follows L-SHADE
on a rotated Rastrigin function.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ioh
import modde

problem = ioh.get_problem(15, instance=1, dimension=10)   # BBOB f15, rotated Rastrigin

## Generation by generation

`step()` runs one generation and returns `False` when the run is over. Between
generations everything can be read: the population (an `(n, d)` array), its
fitness, the population size `lambda_`, the archive, the SHADE memories, ...

In [ ]:
opt = modde.ModularDE(problem, budget=50_000, preset="L-SHADE", seed=1)
rows = []
while opt.step():
    memory_CR = opt.memory_CR                      # NaN marks a terminal slot (CR = 0)
    rows.append((opt.evaluations, opt.best_f - problem.optimum.y, opt.lambda_, len(opt.archive),
                 np.mean(opt.memory_F), np.mean(np.nan_to_num(memory_CR)),
                 opt.population.std(axis=0).mean()))      # diversity: mean spread per coordinate
evals, precision, lam, archive, mF, mCR, spread = map(np.array, zip(*rows))
print(f"{len(rows)} generations, final precision {precision[-1]:.2e}")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
ax[0, 0].semilogy(evals, np.maximum(precision, 1e-10))
ax[0, 0].set_title("best precision f - f*")
ax[0, 1].plot(evals, lam, label="population")
ax[0, 1].plot(evals, archive, label="archive")
ax[0, 1].set_title("sizes")
ax[0, 1].legend()
ax[1, 0].plot(evals, mF, label="F")
ax[1, 0].plot(evals, mCR, label="CR (terminal slots as 0)")
ax[1, 0].set_title("mean of the SHADE memories")
ax[1, 0].legend()
ax[1, 0].set_xlabel("evaluations")
ax[1, 1].semilogy(evals, spread)
ax[1, 1].set_title("diversity: mean std per coordinate")
ax[1, 1].set_xlabel("evaluations")
fig.tight_layout()

The population shrinks linearly (L-SHADE's population size reduction) and the
archive with it. Diversity is computed here from the population array itself;
the logged statistic `popstd` is something else (the standard deviation over
all coordinates of all individuals pooled together, as in 0.0.4).

## Step by step

A generation consists of separate steps (see [Concepts](../concepts.md)).
Each can be called on its own, and the intermediate results inspected:

In [ ]:
problem.reset()              # the problem counts evaluations; start from zero again
opt = modde.ModularDE(problem, budget=5_000, preset="L-SHADE", seed=2)
opt.initialize()

opt.sample_parameters()      # F and CR per individual
opt.mutate()                 # donor vectors
opt.crossover()              # trial vectors
opt.correct_bounds()         # repair coordinates outside the bounds
print("F used (first 5):  ", np.round(opt.F_used[:5], 3))
print("CR used (first 5): ", np.round(opt.CR_used[:5], 3))
print("donors:", opt.donors.shape, "| trials:", opt.trials.shape)
print("share of coordinates taken from the donor:", np.mean(opt.trials == opt.donors).round(3))

opt.evaluate()
opt.select()
print("improved individuals:", len(opt.success["S"]), "of", opt.lambda_)
opt.adapt()                  # archive, memories, population size
print("archive size after adapt:", len(opt.archive))

The state is also writable between steps: for example, you can replace the
population (`opt.population = ...`) or the F values used (`opt.F_used = ...`)
to try your own operators while modde does the rest.

## Logging the state with ioh

To store these values for every evaluation instead, attach an ioh logger that
**watches** the optimizer and set `track_evaluations=True`; see the
[Quickstart](01_quickstart.ipynb) and [Benchmarking](04_benchmarking.ipynb)
notebooks. The attributes that can be watched are `curr_F`, `curr_CR`,
`corrected`, `corr_so_far`, `CS`, `ED`, `popmean`, `popstd`, `F_memory_mean`
and `generation`.